## Topic 4: Indexing & HNSW Explained

### Why Do We Need Indexing?
- Suppose an Email FAQ system stores only 100 document embeddings.
- When a user asks a question, the system compares the query embedding with all 100 vectors.
- This is called **Brute-Force Search**.
- It works well because the dataset is small.

- Now imagine the knowledge base grows to:
    - 100,000 vectors
    - 1 million vectors
    - 100 million vectors

- Comparing the query against every vector becomes very slow.
- As the number of vectors increases, search time also increases.
- To avoid checking every vector, vector databases build an **index**.
- An index is a data structure that helps the database quickly find the most likely matching vectors without comparing every stored vector.
- Instead of searching the entire collection, it intelligently searches only a small portion.
- This makes retrieval much faster.

---

### Exact Search vs Approximate Search

There are two ways to perform vector search.

#### Exact Search

Every stored vector is compared with the query vector.

Advantages

- Always returns the true nearest neighbors.
- Highest retrieval accuracy.

Disadvantages

- Slow for large datasets.
- Search time increases as the dataset grows.

---

#### Approximate Nearest Neighbor (ANN) Search

Instead of checking every vector, the database searches only the most promising candidates.

Advantages

- Extremely fast.
- Scales to millions of vectors.
- Used by almost all production vector databases.

Disadvantages

- May occasionally miss the absolute closest vector.
- Returns results that are very close to the correct answer rather than mathematically perfect.

For RAG applications, this trade-off is acceptable because retrieving a highly relevant chunk is usually sufficient.

---

## What is HNSW?

HNSW stands for **Hierarchical Navigable Small World**.

It is one of the most popular Approximate Nearest Neighbor (ANN) indexing algorithms.

Qdrant uses HNSW by default.

Instead of storing vectors as a simple list, HNSW organizes them as a graph where similar vectors are connected together.

During search, the algorithm follows these connections to quickly reach vectors that are close to the query instead of examining every stored vector.

---

## How HNSW Works

### Step 1

Every document embedding becomes a node inside the graph.

---

### Step 2

Each node is connected to several nearby nodes.

These connections are created when the index is built.

---

### Step 3

When a query arrives, HNSW does not compare it with every vector.

Instead, it starts from one node and follows neighboring nodes that appear increasingly similar to the query.

---

### Step 4

After exploring the most promising region of the graph, it returns the Top-K nearest vectors.

This process requires far fewer comparisons than brute-force search.

---

## Example

Suppose an Email FAQ system stores one million document embeddings.

Without HNSW:

- Compare the query against all one million vectors.

With HNSW:

- Navigate through connected vectors.
- Search only a small portion of the graph.
- Return the nearest matching documents.

The user receives almost the same results but much faster.

---

## Why Does Qdrant Use HNSW?

Qdrant uses HNSW because it provides an excellent balance between speed and retrieval quality.

It offers:

- Very fast search.
- High retrieval accuracy.
- Good scalability.
- Efficient memory usage.
- Excellent production performance.

For most RAG applications, HNSW provides near-exact results while reducing search time dramatically.

---

## Advantages

- Extremely fast similarity search.
- Scales to millions of vectors.
- High retrieval quality.
- Production proven.
- Automatically managed by Qdrant.
- No manual graph management required.

---

## Disadvantages

- Search is approximate rather than perfectly exact.
- Building the index requires additional time.
- Consumes more memory than storing vectors alone.
- Performance depends on index configuration.

---

## Real-World Examples

### Customer Support Chatbot

Millions of FAQ embeddings are stored.

Instead of checking every FAQ, HNSW quickly navigates to similar questions.

---

### Banking Knowledge Base

Employees search thousands of policy documents.

HNSW retrieves relevant policies within milliseconds.

---

### Enterprise Search

Large organizations store millions of documents.

HNSW enables semantic search with very low response times.

---

## Real-World Issues, Edge Cases & Debugging

### Small Dataset

Problem

The dataset contains only a few hundred vectors.

Issue

Brute-force search may actually be faster because there is almost no search overhead.

---

### Large Dataset

Problem

Millions of vectors must be searched.

Issue

Brute-force search becomes too slow.

Solution

Use HNSW indexing.

---

### Embedding Quality

Problem

Search results are poor.

Issue

The problem is often the embedding model rather than HNSW itself.

Improving embeddings usually improves retrieval quality.

---

### Metadata Filtering

Problem

Very strict metadata filters leave only a few matching vectors.

Issue

The search space becomes very small, which may reduce retrieval effectiveness.

---

## Common Mistakes & Production Failures

- Assuming HNSW always returns the exact nearest vector.
- Blaming HNSW when poor embeddings are the real issue.
- Using brute-force search for very large datasets.
- Ignoring memory requirements when storing millions of vectors.
- Expecting indexing to improve poor-quality embeddings.

---

## Lead-Level Interview Questions

### Q1. Why do vector databases build indexes?

**Answer**

Without an index, every search compares the query with every stored vector. As datasets grow, this becomes too slow. An index reduces the number of comparisons, allowing fast similarity search even for millions of vectors.

---

### Q2. Why is Approximate Nearest Neighbor search acceptable for RAG?

**Answer**

RAG does not require the mathematically closest document. It only needs highly relevant documents that help the LLM generate a correct answer. The small loss in accuracy is outweighed by the significant improvement in search speed.

---

### Q3. Why does Qdrant use HNSW?

**Answer**

HNSW provides an excellent balance between retrieval quality, speed, memory usage, and scalability. It has become the default indexing algorithm for many production vector databases.

---

### Q4. When might brute-force search still be preferable?

**Answer**

For very small datasets where search is already fast, brute-force search is simpler and guarantees exact results. The overhead of building an index may not be worthwhile.

---

### Q5. Does HNSW improve embedding quality?

**Answer**

No.

HNSW only makes similarity search faster.

The quality of retrieval still depends primarily on the embedding model. Poor embeddings will produce poor results regardless of the indexing algorithm.

---

## Key Takeaways

- Indexing exists to avoid comparing every vector during search.
- Brute-force search is exact but does not scale well.
- HNSW is an Approximate Nearest Neighbor indexing algorithm.
- Qdrant uses HNSW automatically.
- HNSW provides fast retrieval while maintaining high accuracy.
- Better embeddings improve retrieval quality more than changing the indexing algorithm.

# End-to-End Flow of `qdrant_hnsw.py`

This example demonstrates a complete vector search pipeline using **Qdrant** with **HNSW indexing**.

The program performs the following steps:

1. Creates an in-memory Qdrant database.
2. Creates a collection with HNSW indexing enabled.
3. Converts text chunks into embeddings.
4. Stores embeddings and metadata inside Qdrant.
5. Performs semantic search.
6. Performs filtered semantic search.
7. Demonstrates the effect of different HNSW search parameters.

---

## Step 1: Create Qdrant Client

```python
client = QdrantClient(":memory:")
```

### What Happens?

This creates a Qdrant instance inside the current Python process.

### Theory Used

- Qdrant supports multiple deployment modes.
- `:memory:` mode requires no Docker or server.
- Everything is stored in RAM.
- Data disappears when Python exits.

### What Actually Happens Internally?

Qdrant starts an in-memory database.

At this point:

- No collection exists.
- No vectors exist.
- No HNSW index exists.
- No metadata exists.

Only an empty Qdrant database is running.

---

## Step 2: Load the Embedding Model

```python
model = SentenceTransformer(MODEL_NAME)
```

### What Happens?

The embedding model is loaded into memory.

### Theory Used

Qdrant does not create embeddings.

Embedding generation is always performed outside Qdrant.

The same embedding model must be used for:

- document embeddings
- query embeddings

Otherwise vector dimensions will not match.

### What Actually Happens Internally?

Nothing is stored inside Qdrant yet.

Only the embedding model is ready to generate vectors.

---

## Step 3: Create Collection

```python
create_hnsw_collection()
```

Inside:

```python
client.create_collection(...)
```

### What Happens?

A collection named **fd_hnsw_demo** is created.

### Theory Used

A collection is similar to a SQL table.

It stores:

- vectors
- payload
- IDs

Every vector inside this collection must have:

- 384 dimensions
- cosine similarity

### HNSW Theory Used

This line:

```python
HnswConfigDiff(...)
```

tells Qdrant:

- Build an HNSW index.
- Use `m=16`.
- Use `ef_construction=100`.

You never build the graph yourself.

Qdrant automatically builds and maintains the HNSW graph.

### What Actually Happens Internally?

Only an empty collection is created.

The HNSW structure is initialized but contains no nodes because no vectors have been inserted.

---

## Step 4: Convert Text into Embeddings

```python
embeddings = model.encode(...)
```

### What Happens?

Each text chunk is converted into a 384-dimensional embedding.

Example:

```
Chunk

↓

384-dimensional vector
```

### Theory Used

Embeddings convert semantic meaning into numerical vectors.

Similar meanings produce nearby vectors.

Different meanings produce distant vectors.

### What Actually Happens Internally?

The embedding model generates one vector for every document.

Qdrant is not involved in this step.

---

## Step 5: Create Point Objects

```python
PointStruct(...)
```

### What Happens?

Each document becomes one Qdrant Point.

Every Point contains:

- ID
- Vector
- Payload

Payload stores metadata.

Example:

- text
- source
- doc_type

### Theory Used

A Point is the basic storage unit in Qdrant.

Think of it like a row in a database.

Instead of columns, each row contains:

- embedding
- metadata

### What Actually Happens Internally?

Python creates Point objects.

They still exist only in application memory.

Nothing has been inserted into Qdrant yet.

---

## Step 6: Upsert into Qdrant

```python
client.upsert(...)
```

### What Happens?

All Point objects are inserted into Qdrant.

### Theory Used

Upsert means:

- Insert if new.
- Update if ID already exists.

### What Actually Happens Internally?

For every Point:

- Vector is stored.
- Payload is stored.
- ID is stored.

At the same time, Qdrant automatically updates the HNSW index.

Every vector becomes a node inside the graph.

Connections between nearby vectors are automatically created.

This is exactly where the HNSW graph is built.

You never write graph-building code yourself.

---

## Step 7: Unfiltered Search

```python
search_unfiltered(...)
```

### What Happens?

The query sentence is converted into an embedding.

Qdrant searches the HNSW index.

Top-K nearest vectors are returned.

### Theory Used

The same embedding model must be used for:

- documents
- queries

Otherwise semantic similarity cannot be computed correctly.

HNSW performs Approximate Nearest Neighbor search instead of brute-force search.

### What Actually Happens Internally?

Qdrant:

- receives the query vector
- traverses the HNSW graph
- identifies the nearest vectors
- computes similarity scores
- returns the top matches

The search is performed on the index instead of comparing every stored vector.

---

## Understanding the Output

Output:

```
score=0.5106

Can I withdraw my FD before maturity?
```

### Why is this the Top Result?

The query is:

```
What happens if I close my FD early?
```

The document discusses:

- FD closure
- withdrawal
- penalty

Although the wording is different, the semantic meaning is almost the same.

Therefore the cosine similarity is highest.

---

Output:

```
score=0.4080

Penalty does not apply...
```

### Why is this the Second Result?

This document also discusses:

- FD closure
- penalty

However, it is more specific to death-related exceptions.

Therefore it is slightly less similar.

---

Output:

```
score=0.3879

Interest rate...
```

### Why is this Retrieved?

It contains information related to Fixed Deposits.

Although less relevant, it is still among the closest vectors.

---

## Step 8: Filtered Search

```python
query_filter=Filter(...)
```

### What Happens?

Only vectors whose payload contains:

```
doc_type="faq"
```

are searched.

### Theory Used

Metadata filtering reduces the search space.

Instead of searching every document, Qdrant searches only documents satisfying the filter.

### What Actually Happens Internally?

The HNSW traversal respects the metadata filter.

Policy documents are ignored.

Only FAQ documents are considered valid search candidates.

---

## Step 9: HNSW Search Parameter (ef)

```python
SearchParams(hnsw_ef=64)
```

### What Happens?

This controls how thoroughly HNSW explores the graph.

### Theory Used

Lower `ef`

- faster search
- lower recall

Higher `ef`

- slower search
- higher recall

### Why Doesn't the Output Change?

The collection contains only eight vectors.

The graph is extremely small.

Even with a low `ef`, almost the entire graph is explored.

At millions of vectors, the difference becomes noticeable.

---

## Mapping Theory to Code

### Qdrant In-Memory Mode

Code

```python
QdrantClient(":memory:")
```

Theory

- No server
- No Docker
- Data stored only in RAM

---

### Collection

Code

```python
create_collection()
```

Theory

A collection stores vectors, metadata, and IDs.

---

### HNSW Index

Code

```python
HnswConfigDiff(...)
```

Theory

Qdrant automatically builds an HNSW graph.

---

### Embeddings

Code

```python
model.encode()
```

Theory

Text is converted into numerical vectors.

---

### Point

Code

```python
PointStruct(...)
```

Theory

A Point consists of:

- ID
- Vector
- Payload

---

### Insert

Code

```python
upsert()
```

Theory

Stores vectors and simultaneously updates the HNSW index.

---

### Semantic Search

Code

```python
query_points()
```

Theory

Qdrant traverses the HNSW graph to retrieve the nearest vectors.

---

### Metadata Filtering

Code

```python
Filter(...)
```

Theory

Only vectors matching the payload conditions participate in the search.

---

### Search Quality

Code

```python
SearchParams(hnsw_ef=...)
```

Theory

Controls the trade-off between search speed and retrieval accuracy.

### Understanding `m` and `ef_construction`

During collection creation, the following HNSW configuration is passed to Qdrant:

```python
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=100,
    ),
)
```

These values **do not build the HNSW graph immediately**.

Instead, they tell Qdrant **how to build the HNSW graph later** when vectors are inserted using `upsert()`.

---

### What is `m`?

`m` controls **how many neighboring nodes each vector keeps inside the HNSW graph.**

Think of it as:

> **How many nearby vectors should each vector remain connected to?**

When a new vector is inserted, Qdrant connects it to only a limited number of nearby vectors instead of connecting it to every vector in the collection.

In this example:

```python
m = 16
```

This means each node will maintain approximately **16 neighbor connections** (per HNSW layer).

#### Why is this needed?

If every vector connected to every other vector:

- Memory usage would become extremely large.
- Index construction would become very slow.
- Search would become inefficient.

By limiting the number of connections, HNSW remains efficient while still providing excellent search quality.

#### Small `m`

- Fewer graph connections.
- Lower memory usage.
- Faster index construction.
- Slightly lower search recall.

#### Large `m`

- More graph connections.
- Higher memory usage.
- Slower index construction.
- Better search recall.

---

### What is `ef_construction`?

When a new vector is inserted, Qdrant must decide:

> **Which nearby vectors should this new vector connect to?**

To make this decision, it first searches existing vectors to find good candidate neighbors.

`ef_construction` controls **how thoroughly Qdrant performs this search while building the graph.**

Think of it as:

> **How many candidate neighbors should Qdrant examine before choosing the final connections?**

In this example:

```python
ef_construct = 100
```

Qdrant examines approximately **100 candidate nodes** before selecting the best neighboring vectors.

#### Why is this needed?

Suppose Qdrant examines only a few nearby vectors.

It may miss better neighbors, resulting in a lower-quality graph.

Examining more candidates takes longer but usually produces a much better HNSW index.

#### Small `ef_construction`

- Faster index construction.
- Lower-quality graph.

#### Large `ef_construction`

- Slower index construction.
- Higher-quality graph.
- Better future search accuracy.

---

### Relationship Between `m` and `ef_construction`

These two parameters work together during index construction.

`ef_construction` determines **how many candidate neighbors Qdrant evaluates.**

`m` determines **how many of those neighbors are finally connected to the new node.**

For this code:

```python
m = 16
ef_construct = 100
```

Qdrant approximately performs the following process for every new vector:

1. A new vector is inserted.
2. Qdrant searches approximately 100 candidate neighboring vectors.
3. It identifies the best neighboring vectors.
4. It creates approximately 16 graph connections.
5. The HNSW graph is updated automatically.

You never write any graph-building code yourself. Qdrant manages the entire HNSW index internally.

---

### Are These Parameters Used During Search?

No.

Both `m` and `ef_construction` are **index-building parameters**.

They are only used while constructing the HNSW graph.

During searching, Qdrant uses a different parameter:

```python
SearchParams(
    hnsw_ef=64
)
```

`hnsw_ef` controls **how thoroughly Qdrant explores the already-built graph during a search.**

---

### Summary

- `m` decides **how many neighbor connections each node keeps** in the HNSW graph.
- `ef_construction` decides **how thoroughly Qdrant searches for good neighbors while building the graph.**
- Both parameters affect **index quality**, not query execution.
- Once the HNSW graph has been built, these values remain fixed until the index is rebuilt.
- Search behavior is controlled separately using `hnsw_ef`.

### What is `hnsw_ef`?

After the HNSW graph has been built, Qdrant uses that graph whenever a search query arrives.

During search, Qdrant does **not** compare the query with every stored vector.

Instead, it navigates through the HNSW graph to find the nearest vectors.

The parameter `hnsw_ef` controls **how thoroughly Qdrant explores the graph during a search.**

Think of it as:

> **How many candidate nodes should Qdrant examine before returning the final search results?**

---

### Where is it Used?

In the code:

```python
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
    search_params=SearchParams(
        hnsw_ef=64
    ),
)
```

Unlike `m` and `ef_construction`, which are used while **building** the HNSW graph, `hnsw_ef` is used **every time a search is performed.**

---

### Why is it Needed?

Suppose the HNSW graph contains one million vectors.

When a query arrives, Qdrant starts navigating through the graph.

The question is:

> **How long should Qdrant continue exploring before returning the answer?**

That is controlled by `hnsw_ef`.

A larger value tells Qdrant to explore more of the graph.

A smaller value tells Qdrant to stop searching earlier.

---

### Small `hnsw_ef`

Example:

```python
SearchParams(
    hnsw_ef=10
)
```

Characteristics:

- Faster search.
- Fewer graph nodes explored.
- Lower CPU usage.
- Slightly lower recall.
- May miss the best matching vector.

---

### Large `hnsw_ef`

Example:

```python
SearchParams(
    hnsw_ef=200
)
```

Characteristics:

- More graph nodes explored.
- Slower search.
- Higher CPU usage.
- Better recall.
- More likely to find the true nearest neighbors.

---

### Why Doesn't the Output Change in This Demo?

Your collection contains only **8 vectors**.

Whether `hnsw_ef` is 10 or 200, Qdrant can easily explore almost the entire graph.

Therefore both searches return the same results.

Example:

```python
search_unfiltered(..., ef=10)

search_unfiltered(..., ef=200)
```

Both produce:

```
Senior citizens receive an additional 0.5 percent interest...
The FD interest rate for 24 months is 7.1 percent...
```

This is expected because the dataset is extremely small.

With millions of vectors, the difference becomes noticeable.

---

### Relationship Between `m`, `ef_construction`, and `hnsw_ef`

These three parameters control different stages of HNSW.

#### `m`

Controls:

- Number of neighbor connections each node keeps.

Used:

- While building the HNSW graph.

---

#### `ef_construction`

Controls:

- How thoroughly Qdrant searches for good neighbors while building the graph.

Used:

- While building the HNSW graph.

---

#### `hnsw_ef`

Controls:

- How thoroughly Qdrant explores the existing graph during a search.

Used:

- Every time a query is executed.

---

### Simple Interview Memory Trick

- **`m`** → How many neighbors each node keeps.
- **`ef_construction`** → How much effort Qdrant spends building the graph.
- **`hnsw_ef`** → How much effort Qdrant spends searching the graph.

---

### Summary

- `m` controls the **graph structure**.
- `ef_construction` controls the **quality of graph construction**.
- `hnsw_ef` controls the **quality of graph traversal during search**.
- `m` and `ef_construction` are fixed after the index is built.
- `hnsw_ef` can be changed for every query without rebuilding the index.

In [1]:
"""
qdrant_hnsw.py
----------------
Demonstrates HNSW index parameter control in Qdrant.
Uses :memory: mode -- no Docker required.

Install: pip install qdrant-client sentence-transformers
"""

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,       # tells Qdrant how to measure closeness between vectors
    VectorParams,   # defines the vector size and distance metric for a collection
    HnswConfigDiff, # lets us override the default HNSW index parameters
    PointStruct,    # the object that holds one point: id + vector + payload
    Filter,         # wraps one or more conditions for a filtered search
    FieldCondition, # one condition: "this payload field must match this value"
    MatchValue,     # the "must equal this exact value" part of a FieldCondition
    SearchParams,   # lets us control ef (search-time recall vs. speed trade-off)
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_hnsw_demo"
VECTOR_SIZE = 384       # must match the embedding model's output dimension
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# :memory: means no Docker, no server -- everything lives in this Python process
client = QdrantClient(":memory:")

# load the embedding model once -- reused for every encode() call below
model = SentenceTransformer(MODEL_NAME)

# small set of sample chunks -- in a real pipeline these come from chunker.py
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to the death of the depositor.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.", "source": "FD_Policy", "doc_type": "policy"},
]


def create_hnsw_collection(m: int = 16, ef_construction: int = 100) -> None:
    # check what collections already exist so we don't crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if it exists -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,       # every vector in this collection must be 384 floats
            distance=Distance.COSINE,  # cosine similarity -- right choice for normalized vectors
        ),
        hnsw_config=HnswConfigDiff(
            m=m,                    # edges per node -- higher = better recall, more memory
            ef_construct=ef_construction,  # build-time search depth -- higher = better index quality
        ),
    )
    print(f"Created collection with HNSW m={m}, ef_construction={ef_construction}")


def upsert_chunks() -> None:
    # extract just the text strings so we can embed them all in one batch call
    texts = [c["text"] for c in CHUNKS]

    # embed all texts at once -- normalize_embeddings=True makes dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build a PointStruct for each chunk: id + vector + payload (metadata)
    points = [
        PointStruct(
            id=i,                           # simple integer ID -- fine for a demo
            vector=embeddings[i].tolist(),  # numpy array -> plain Python list for serialization
            payload={
                "text": CHUNKS[i]["text"],      # store original text so search results are self-contained
                "source": CHUNKS[i]["source"],  # which source file this chunk came from
                "doc_type": CHUNKS[i]["doc_type"],  # used for filtering later
            },
        )
        for i in range(len(CHUNKS))
    ]

    # upsert = insert if new, replace if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    # confirm how many points are now in the collection
    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {info.points_count} points")


def search_unfiltered(query: str, k: int = 3, ef: int = 128) -> None:
    # embed the query with the same model used at ingest time -- must always match
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # query_points replaces the old client.search() which was removed in qdrant-client >= 1.9
    # .points unwraps the result object to get the actual list of scored hits
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,         # the embedded query vector to search against
        limit=k,                    # return the top-k most similar points
        search_params=SearchParams(
            hnsw_ef=ef              # higher ef = more thorough search = better recall, slower
        ),
        with_payload=True,          # include the payload (text, source, doc_type) in results
    ).points                        # .points extracts the list from the response wrapper

    print(f"\nUnfiltered search: '{query}' (ef={ef})")
    for r in results:
        # r.score is the cosine similarity -- closer to 1.0 = more similar
        print(f"  score={r.score:.4f} | {r.payload['text'][:70]}")


def search_filtered(query: str, doc_type: str, k: int = 3) -> None:
    # same embedding step as unfiltered search -- model must always be the same
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[                          # ALL conditions in must[] must be true
                FieldCondition(
                    key="doc_type",         # the payload field to filter on
                    match=MatchValue(value=doc_type),  # must equal this exact value
                )
            ]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nFiltered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create the collection with explicit HNSW settings
create_hnsw_collection(m=16, ef_construction=100)

# step 2: embed and store all sample chunks
upsert_chunks()

# step 3: search across all doc types -- no filter
search_unfiltered("What happens if I close my FD early?", k=3, ef=64)

# step 4: same query but only look inside FAQ documents
search_filtered("What happens if I close my FD early?", doc_type="faq", k=2)

# step 5: different query, only look inside policy documents
search_filtered("penalty for early closure", doc_type="policy", k=3)

# step 6: compare low ef vs high ef on the same query
# at 8 points the scores will be identical -- the difference shows at scale
print("\n--- ef comparison (difference visible at scale, not at 8 points) ---")
search_unfiltered("senior citizen interest rate", k=2, ef=10)   # fast, lower recall at scale
search_unfiltered("senior citizen interest rate", k=2, ef=200)  # slower, higher recall at scale


Created collection with HNSW m=16, ef_construction=100
Upserted 8 points

Unfiltered search: 'What happens if I close my FD early?' (ef=64)
  score=0.5106 | Can I withdraw my FD before maturity? Yes, subject to a penalty.
  score=0.4080 | This penalty does not apply if the FD is closed due to the death of th
  score=0.3879 | The FD interest rate for 24 months is 7.1 percent per annum.

Filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

Filtered search (doc_type='policy'): 'penalty for early closure'
  score=0.6175 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.4741 | [policy] This penalty does not apply if the FD is closed due to the d
  score=0.0928 | [policy] TDS is deducted at source if interest income exceeds Rs 40,0

--- ef comparison (difference visible at 